# AdaBoost (Adaptive Boosting) Classifier

**AdaBoost**, short for **Adaptive Boosting**, is one of the foundational boosting algorithms in machine learning. It belongs to the class of **sequential ensemble methods**, where models are trained one after another. The primary objective of AdaBoost is to combine multiple "weak learners" (classifiers that perform slightly better than random guessing) to build a single, highly accurate "strong classifier."

In these detailed notes, we will cover:
1. **Core Concept:** Weak learners, Decision Stumps, and Stage-wise Additive Modeling.
2. **The AdaBoost Algorithm (Step-by-Step):** Initial weights, error calculation, model influence (alpha), weight updating, and resampling.
3. **Mathematical Formulations:** Error rate, alpha calculation, and prediction sign voting.
4. **Geometric Intuition:** Visualizing decision boundary refinement.
5. **Bagging vs. Boosting:** A comparative study of type of models, training processes, and weighting systems.
6. **Key Hyperparameters & Regularization:** Estimators, learning rate, and shrinkage.
7. **Practical Python Demos:**
   - **Demo A:** Step-by-Step AdaBoost implementation from scratch on a toy dataset.
   - **Demo B:** Visualizing boundary refinement across different estimators and learning rates.
   - **Demo C:** Hyperparameter tuning using `GridSearchCV` on real-world data.
   - **Summary Checklist** for AdaBoost.

## 1. Core Concepts & Prerequisites

### Weak Learners
A **Weak Learner** is a machine learning model that performs only slightly better than random guessing (i.e., its accuracy is just over $50\%$ for binary classification). AdaBoost leverages the concept that combining many weak learners can mathematically produce an extremely accurate ensemble.

### Decision Stumps
The standard weak learner used in AdaBoost is a **Decision Stump**. A Decision Stump is a Decision Tree with a **maximum depth of 1**. It consists of a single split node and two leaf nodes, making classification decisions based on a single feature threshold (e.g., $X_1 > \theta$).

### Classification Target Representation
Mathematically, AdaBoost represents binary classification targets as:

$$y_i \in \{-1, +1\}$$

This target representation simplifies the mathematical formulation of weight updates and predictions.

### Stage-wise Additive Modeling
AdaBoost is a **stage-wise additive model**. This means the ensemble is built incrementally. Instead of training all models simultaneously, it trains one weak learner at a time, evaluates its errors, updates training sample weights, and then adds a new learner that focuses on correcting the preceding mistakes.

## 2. The AdaBoost Algorithm: Step-by-Step

Let $S = \{(x_1, y_1), \dots, (x_N, y_N)\}$ be a training set of $N$ samples, where $y_i \in \{-1, +1\}$.

### Step 1: Weight Initialization
Initialize the weights of all training instances equally. Since there are $N$ instances, each instance $x_i$ gets an initial weight of:

$$w_i = \frac{1}{N}$$

---

### Step 2: The Iterative Boosting Loop
For $t = 1$ to $T$ (where $T$ is the number of estimators):

#### A. Train the Weak Learner
Fit a decision stump $h_t(X)$ to the training data using the current sample weights $w_i$. The algorithm selects the feature and split point that minimizes impurity (e.g., Gini/Entropy).

#### B. Calculate Total Error ($\epsilon_t$)
Evaluate the stump's performance. The total error $\epsilon_t$ is the sum of the weights of all misclassified instances:

$$\epsilon_t = \sum_{i=1}^{N} w_i \cdot \mathbb{I}(y_i \neq h_t(x_i))$$

*(Where $\mathbb{I}(\cdot)$ is the indicator function, returning $1$ if true and $0$ if false. Note that if $\epsilon_t \ge 0.5$, training terminates early or restarts, as a weak learner must perform better than random guessing.)*

#### C. Calculate Model Influence / Weight ($\alpha_t$)
Compute the weight/influence $\alpha_t$ of the current classifier in the final prediction:

$$\alpha_t = \frac{1}{2} \ln\left( \frac{1 - \epsilon_t}{\epsilon_t} \right)$$

*   **Behavior of $\alpha_t$:** 
    *   If a model has **zero error** ($\epsilon_t \to 0$), then $\alpha_t \to +\infty$ (highly influential).
    *   If a model has **$50\%$ error** ($\epsilon_t = 0.5$, random guess), then $\alpha_t = 0$ (no influence).
    *   If a model has **high error** ($\epsilon_t > 0.5$), then $\alpha_t < 0$ (reverses predictions).

#### D. Update Sample Weights
Increase the importance of misclassified samples and decrease the importance of correctly classified samples:

$$w_i \leftarrow w_i \cdot e^{-\alpha_t \cdot y_i \cdot h_t(x_i)}$$

This works because:
*   If $x_i$ is **correctly classified**, $y_i$ and $h_t(x_i)$ have the same sign, so $y_i \cdot h_t(x_i) = 1$. The weight becomes $w_i \cdot e^{-\alpha_t}$ (decreased).
*   If $x_i$ is **misclassified**, $y_i$ and $h_t(x_i)$ have opposite signs, so $y_i \cdot h_t(x_i) = -1$. The weight becomes $w_i \cdot e^{\alpha_t}$ (increased/boosted).

#### E. Normalize Sample Weights
Normalize the weights so they sum up to $1.0$ (ensuring a valid probability distribution):

$$w_i \leftarrow \frac{w_i}{\sum_{j=1}^{N} w_j}$$

#### F. Resampling
To prepare the dataset for the next iteration, the algorithm performs resampling with replacement. It creates ranges based on the normalized weights. The algorithm generates random numbers between $0$ and $1$ to select rows. Since misclassified rows have larger weights, they have a larger probability of being selected multiple times (upsampling), forcing the next weak learner to focus on correcting those errors.

---

### Step 3: Final Prediction
To classify a new data point $X$, AdaBoost calculates the weighted sum of the predictions from all $T$ decision stumps, and takes the sign ($+$ or $-$) of the result:

$$F(X) = \text{sign}\left( \sum_{t=1}^{T} \alpha_t h_t(X) \right)$$

If the sum is positive, the predicted class is $+1$. If negative, the predicted class is $-1$.

## 3. Geometric Intuition

Since decision stumps are restricted to a single axis-aligned split, their individual decision boundaries are simple straight vertical or horizontal lines (in 2D space).

### Boundary Refinement
*   **Stump 1:** Creates a single split. It classifies some points correctly, but misses others.
*   **Stump 2:** Trained on the re-weighted dataset, it creates a new split that specifically targets the mistakes of Stump 1.
*   **Ensembling:** When we take a weighted sum of these orthogonal lines, the boundaries begin to stack. 

As we add more estimators, the combination of multiple simple splits creates a **highly complex, non-linear decision boundary** that wraps around the data distribution, optimizing class separation.

## 4. Bagging vs. Boosting Comparison

Bagging and Boosting are both ensemble techniques, but they rely on fundamentally different mechanics:

| Feature | Bagging (Parallel Ensembling) | Boosting (Sequential Ensembling) |
| :--- | :--- | :--- |
| **Base Learner Type** | **Low Bias, High Variance** models (e.g., fully grown, unconstrained Decision Trees). | **High Bias, Low Variance** models (e.g., shallow Decision Trees / Decision Stumps). |
| **Learning Strategy** | **Parallel:** All base estimators are trained simultaneously and independently on bootstrap samples. | **Sequential:** Estimators are trained one after another. Each model corrects the errors of its predecessor. |
| **Model Weighting** | **Democratic:** Every model has equal voting weight ($1/M$) in the final prediction. | **Weighted:** Each model gets a custom weight ($\alpha_t$) based on its accuracy. More accurate models have more influence. |
| **Sample Selection** | Independent bootstrap sampling. | Adaptive sample re-weighting (errors get higher weights). |
| **Optimization Goal** | Focuses on **reducing variance** (overfitting). | Focuses on **reducing bias** (underfitting) and variance. |
| **Sensitivity** | Robust to outliers and noise. | Can be sensitive to noise/outliers (sequential error-chasing risk). |

## 5. Hyperparameters & Regularization

In Scikit-Learn, `AdaBoostClassifier` is governed by the following key hyperparameters:

*   **`estimator`** (formerly `base_estimator`): The weak learner to clone. Defaults to a `DecisionTreeClassifier(max_depth=1)`. The estimator **must** support training with sample weights.
*   **`n_estimators`:** The maximum number of weak learners to train sequentially. If a perfect training fit is achieved, training stops early.
*   **`learning_rate`:** A scaling factor applied to the contribution ($\alpha_t$) of each estimator. Acts as a regularization parameter.
*   **`algorithm`:** `'SAMME'` or `'SAMME.R'`. `'SAMME.R'` (Real) is the default; it leverages probability estimates to converge faster and with lower test error than `'SAMME'` (Discrete).

### Regularization via Shrinkage
To prevent overfitting in boosting, we balance `n_estimators` and `learning_rate`:
*   **The Learning Rate ($\eta$) Shrinkage:** The weight coefficient $\alpha_t$ calculated for each stump is scaled down:
    $$\alpha_t \leftarrow \eta \cdot \alpha_t$$
*   By reducing the learning rate (e.g., $\eta = 0.1$), the impact of each single stump's prediction is reduced. This slows down the training process, causing the sample weights to update in smaller increments.
*   *Trade-off:* A lower learning rate requires a **higher number of estimators** (`n_estimators`), but it results in a **smoother, more generalized decision boundary** and reduces the risk of overfitting.

## 6. Code Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

# Set visualization contexts
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 7. Demo A: AdaBoost Step-by-Step From Scratch

We will implement the mathematical steps of AdaBoost from scratch on a small toy dataset (10 rows, 2 features) to trace sample weight adjustments, total error calculations, and model weights (alphas).

In [ ]:
# 1. Create a toy dataset with 10 rows and 2 features
np.random.seed(42)
X_toy = np.random.randn(10, 2)
# Assign binary labels: +1 or -1
y_toy = np.where(X_toy[:, 0] + X_toy[:, 1] > 0, 1, -1)

# Initialize sample weights equally
N = len(X_toy)
w = np.ones(N) / N

print("Initial Sample Weights (w_i):")
print(w)
print("-"*55)

# Dataframe to track updates
df_track = pd.DataFrame({
    'X1': X_toy[:, 0],
    'X2': X_toy[:, 1],
    'y': y_toy,
    'w_initial': w
})

# Let's run 3 boosting iterations manually
alphas = []
models = []

for t in range(3):
    # A. Train Decision Stump (depth=1)
    stump = DecisionTreeClassifier(max_depth=1, random_state=42)
    # We pass sample_weight to the fit method to train based on current weights
    stump.fit(X_toy, y_toy, sample_weight=w)
    y_pred = stump.predict(X_toy)
    
    # B. Calculate total weighted error
    misclassified = (y_toy != y_pred)
    epsilon_t = np.sum(w[misclassified])
    
    # C. Calculate model weight (alpha)
    # Add a small offset to epsilon to prevent division by zero
    epsilon_t = max(epsilon_t, 1e-10)
    alpha_t = 0.5 * np.log((1.0 - epsilon_t) / epsilon_t)
    
    # D. Update weights
    w = w * np.exp(-alpha_t * y_toy * y_pred)
    
    # E. Normalize weights
    w = w / np.sum(w)
    
    # Record values
    alphas.append(alpha_t)
    models.append(stump)
    
    df_track[f'pred_t{t+1}'] = y_pred
    df_track[f'w_after_t{t+1}'] = w
    
    print(f"Iteration {t+1}:")
    print(f"  Weighted Error (epsilon): {epsilon_t:.4f}")
    print(f"  Model Weight (alpha):     {alpha_t:.4f}")
    print("-"*55)

# Check track dataframe
print("Sample Weights and Predictions Tracker:")
print(df_track.to_string(index=False))
print("-"*55)

# F. Predict using custom aggregator
raw_sum = np.zeros(N)
for alpha, model in zip(alphas, models):
    raw_sum += alpha * model.predict(X_toy)
final_pred = np.sign(raw_sum)

print(f"Weighted sum outputs: \n{raw_sum}")
print(f"Final predictions (sign function): \n{final_pred}")
print(f"True targets:                       \n{y_toy}")
print(f"Scratch accuracy: {accuracy_score(y_toy, final_pred):.4f}")

## 8. Demo B: Visualizing Boundary Refinement & Learning Rates

We will evaluate AdaBoost boundaries on 2D synthetic "moons" data across different numbers of estimators ($T = 1, 5, 20, 100$) and investigate the smoothing effect of the learning rate parameter.

In [ ]:
# Generate classification moons dataset
X_moons, y_moons = make_moons(n_samples=250, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_moons, y_moons, test_size=0.3, random_state=42)

# Helper function to plot decision boundaries
def plot_adaboost_boundary(model, X, y, ax, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#3498db', edgecolor='k', s=35, label='Class 0')
    ax.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#e74c3c', edgecolor='k', s=35, label='Class 1')
    
    acc = accuracy_score(y_test, model.predict(X_test))
    ax.set_title(f"{title}\nTest Acc: {acc:.4f}", fontsize=11, fontweight='bold')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.legend(frameon=True, facecolor='white')

# 1. Plot boundary refinement as we add estimators (n_estimators = 1, 5, 20, 100)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, n_est in enumerate([1, 5, 20, 100]):
    ada = AdaBoostClassifier(n_estimators=n_est, learning_rate=1.0, random_state=42)
    ada.fit(X_train, y_train)
    plot_adaboost_boundary(ada, X_train, y_train, axes[idx], f"AdaBoost ({n_est} Estimators)")

plt.suptitle("AdaBoost Boundary Refinement: Stacking Axis-Aligned Splits", fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# 2. Plot boundary comparison: High Learning Rate vs. Regularized Learning Rate
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# High learning rate, few estimators
ada_fast = AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=42)
ada_fast.fit(X_train, y_train)
plot_adaboost_boundary(ada_fast, X_train, y_train, axes[0], "AdaBoost (n_est=50, Learning Rate=1.0)")

# Regularized learning rate, more estimators (Shrinkage)
ada_smooth = AdaBoostClassifier(n_estimators=500, learning_rate=0.1, random_state=42)
ada_smooth.fit(X_train, y_train)
plot_adaboost_boundary(ada_smooth, X_train, y_train, axes[1], "AdaBoost (n_est=500, Learning Rate=0.1) - Regularized")

plt.tight_layout()
plt.show()

## 9. Demo C: Hyperparameter Tuning via GridSearchCV

We will demonstrate optimizing the trade-off between `n_estimators` and `learning_rate` using cross-validation on synthetic classification data.

In [ ]:
# 1. Generate classification dataset
X_clf, y_clf = make_classification(n_samples=500, n_features=12, n_informative=8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

# 2. Train baseline AdaBoost model
base_ada = AdaBoostClassifier(random_state=42)
base_ada.fit(X_train, y_train)
base_acc = accuracy_score(y_test, base_ada.predict(X_test))

# 3. Run Grid Search CV
param_grid = {
    'n_estimators': [50, 100, 200, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
    'algorithm': ['SAMME']  # Newer scikit-learn uses SAMME, older supports SAMME.R
}

grid = GridSearchCV(
    estimator=AdaBoostClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train, y_train)

tuned_ada = grid.best_estimator_
tuned_acc = accuracy_score(y_test, tuned_ada.predict(X_test))

print("GridSearchCV Results:")
print("="*55)
print(f"Baseline AdaBoost Accuracy: {base_acc:.4f}")
print(f"Tuned AdaBoost Accuracy:    {tuned_acc:.4f}")
print(f"Best Hyperparameters:       {grid.best_params_}")
print("="*55)
print()
print("Classification Report (Tuned Model):")
print(classification_report(y_test, tuned_ada.predict(X_test)))

## Summary Checklist for AdaBoost

1.  **Core Intuition:** Train a sequence of **Weak Learners** (usually **Decision Stumps** of depth=1) that build on top of each other sequentially.
2.  **Focus on Errors:** Misclassified points from the current iteration get their weights boosted (increased), forcing the next stump to focus on correcting those errors.
3.  **Understand Sample Weight Updates:** Correctly classified points get their weights reduced ($w_i \leftarrow w_i \cdot e^{-\alpha_t}$), while misclassified points get their weights increased ($w_i \leftarrow w_i \cdot e^{\alpha_t}$).
4.  **Influence Assignment (Alpha):** Each stump gets assigned a custom model weight $\alpha_t = \frac{1}{2} \ln(\frac{1 - \epsilon_t}{\epsilon_t})$ based on its performance. Models with lower errors get exponentially higher influence.
5.  **Final Prediction:** Sum the predictions of all stumps weighted by their respective alpha values, and take the sign: $F(X) = \text{sign}(\sum \alpha_t h_t(X))$.
6.  **Bagging vs. Boosting:**
    *   *Bagging:* Parallel training, low-bias/high-variance base estimators (deep trees), equal model weights.
    *   *Boosting:* Sequential training, high-bias/low-variance base estimators (shallow trees), weighted model contributions.
7.  **Optimize regularizations:** Balance `learning_rate` and `n_estimators`. Implement **Shrinkage** by reducing learning rate (e.g. to $0.1$) and increasing estimators (e.g. to $500$) to yield smoother, generalized decision boundaries.
8.  **Resampling Mechanics:** To feed data to the next stage, the algorithm resamples rows based on updated weight probabilities, ensuring misclassified rows are upsampled.